# 🏔️ ALTAY LLM — SON DENEME
Hata çıkarsa **TEKRAR ÇALIŞTIR** yeter.

1. Ctrl+F9 bas
2. Token'ı yapıştır
3. ~1.5 saat bekle

In [ ]:
# ============================================================
# TEK HÜCRE - HER ŞEY BURADA
# ============================================================
import os, sys, json, gc, random, torch
from getpass import getpass

print("🏔️ ALTAY LLM basliyor...")
print("GPU kontrol ediliyor...")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
if not torch.cuda.is_available():
    print("❌ GPU YOK! Ustteki menuden: Calisma Zamani > Turunu degistir > T4 GPU sec > Kaydet")
    print("  Sonra bu hucreyi TEKRAR CALISTIR (sol ustteki ▶️)")
    sys.exit()

print(f"✅ GPU: {torch.cuda.get_device_name()}")

# --- KURULUM ---
print("📦 Kutuphaneler yukleniyor... (2 dk)")
!pip install -q transformers datasets accelerate peft bitsandbytes huggingface_hub

from huggingface_hub import login, HfApi, create_repo
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer,
    BitsAndBytesConfig, DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset, Dataset

# --- TOKEN ---
try:
    HF_TOKEN = getpass('🤗 Hugging Face token: ')
    login(token=HF_TOKEN)
    print("✅ Token alindi")
except Exception as e:
    print(f"❌ Token hatasi: {e}")
    sys.exit()

# --- MODEL ---
print("🚀 Qwen 2.5 7B yukleniyor... (3 dk)")
try:
    model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-3B",
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        ),
        device_map="auto", trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=16, lora_dropout=0,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        bias="none", task_type="CAUSAL_LM",
    ))
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B", trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    m = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    print(f"✅ Model hazir! {m:.1f}M parametre")
except Exception as e:
    print(f"❌ Model yuklenemedi: {e}")
    sys.exit()

# --- VERI ---
print("📚 Veri yukleniyor... (2 dk)")
try:
    oa = load_dataset("OpenAssistant/oasst1", split="train", streaming=True)
    threadlar = {}
    for item in oa:
        t = item["message_tree_id"]
        if t not in threadlar: threadlar[t] = []
        threadlar[t].append(item)
        if len(threadlar) >= 4000: break

    SISTEM = "Sen Altay LLM'sin. Turkce ve Ingilizce yardimci bir asistansin."
    veri = []
    for tid, msgs in threadlar.items():
        msgs.sort(key=lambda x: x["created_date"])
        dil = msgs[0].get("lang","en")
        if dil not in ["tr","en"]: continue
        if dil=="en" and len(veri)>400: continue
        conv = [{"role":"system","content":SISTEM}]
        for m in msgs:
            conv.append({"role":"assistant" if m["role"]=="assistant" else "user", "content":m["text"]})
        if len(conv)>=2:
            veri.append(tokenizer.apply_chat_template(conv, tokenize=False))
        if len(veri)>=1500: break

    random.shuffle(veri)
    train = Dataset.from_dict({"text": veri})

    def tok(ornek):
        return tokenizer(ornek["text"], truncation=True, padding=False, max_length=1024)
    train = train.map(tok, batched=True, remove_columns=["text"])
    print(f"✅ {len(train)} ornek")
except Exception as e:
    print(f"❌ Veri hatasi: {e}")
    sys.exit()

# --- EGITIM ---
print("🔥 EGITIM BASLIYOR! ~45 dk")
print("Colab acik kalsin, ben bitince haber veririm.")
torch.cuda.empty_cache()
gc.collect()
try:
    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir="./cikti",
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            learning_rate=2e-4,
            warmup_steps=25,
            lr_scheduler_type="cosine",
            logging_steps=10,
            save_strategy="no",
            bf16=torch.cuda.is_bf16_supported(),
            fp16=not torch.cuda.is_bf16_supported(),
            gradient_checkpointing=True,
            dataloader_num_workers=0,
            optim="adamw_8bit",
            report_to="none",
            max_steps=300,
        ),
        train_dataset=train,
        data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8),
    )
    trainer.train()
    print("✅ EGITIM TAMAM!")
except Exception as e:
    print(f"❌ Egitim hatasi: {e}")
    print("Tekrar Ctrl+F9 yap, genelde ikincide calisir.")
    sys.exit()

# --- KAYDET ---
print("💾 Hugging Face'e kaydediliyor...")
try:
    model.save_pretrained("./altay-son")
    tokenizer.save_pretrained("./altay-son")
    api = HfApi()
    create_repo("cntzcn10/altay-llm", exist_ok=True)
    api.upload_folder(repo_id="cntzcn10/altay-llm", folder_path="./altay-son", path_in_repo=".")
    print(f"✅ Model yayinda!")
    print(f"   https://huggingface.co/cntzcn10/altay-llm")
except Exception as e:
    print(f"❌ Kayit hatasi: {e}")

# --- TEST ---
print("\n🧪 Test ediliyor...")
try:
    model.eval()
    for soru in ["Merhaba, kimsin?", "Turkiye'nin baskenti neresi?"]:
        msg = [{"role":"system","content":SISTEM},{"role":"user","content":soru}]
        girdi = tokenizer.apply_chat_template(msg, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
        with torch.no_grad():
            cikti = model.generate(girdi, max_new_tokens=100, temperature=0.7)
        cevap = tokenizer.decode(cikti[0][girdi.shape[1]:], skip_special_tokens=True)
        print(f"\n🧑 {soru}")
        print(f"🤖 {cevap}")
    print("\n🎉🎉🎉 ALTAY LLM HAZIR!")
except Exception as e:
    print(f"Test hatasi: {e}")
